# Digital Clone

Turn on GPU: **Settings → Accelerator → GPU T4 x2** (or P100), then attach
the private dataset containing `voice_reference.wav` and `face_clip.mp4`
(produced locally by `scripts/extract_voice_sample.py` and
`scripts/prepare_face_clip.py`) under **Add Input**.

Edit `SCRIPT_TEXT` below, then **Run All**. Output lands at
`/kaggle/working/output.mp4`.

In [ ]:
# --- Parameters ---
SCRIPT_TEXT = (
    "Hi, this is my digital clone. Everything you hear was typed, not recorded."
)

DATASET_DIR = "/kaggle/input/digital-clone-assets"  # rename to match your dataset
VOICE_REFERENCE = f"{DATASET_DIR}/voice_reference.wav"
FACE_CLIP = f"{DATASET_DIR}/face_clip.mp4"

WORK_DIR = "/kaggle/working"
GENERATED_SPEECH = f"{WORK_DIR}/generated_speech.wav"
OUTPUT_VIDEO = f"{WORK_DIR}/output.mp4"

## 1. Install dependencies

In [ ]:
!pip install -q TTS
!git clone -q https://github.com/Rudrabha/Wav2Lip.git /kaggle/working/Wav2Lip
!pip install -q -r /kaggle/working/Wav2Lip/requirements.txt || true

import os
os.makedirs("/kaggle/working/Wav2Lip/checkpoints", exist_ok=True)
os.makedirs("/kaggle/working/Wav2Lip/face_detection/detection/sfd", exist_ok=True)

## 2. Fetch Wav2Lip checkpoints

The upstream Google Drive links are frequently dead. If the download below
fails, attach a Kaggle dataset with `wav2lip_gan.pth` and `s3fd.pth` as a
second input and point `CKPT_SRC` / `S3FD_SRC` at it instead.

In [ ]:
import shutil, urllib.request

CKPT_SRC = None  # e.g. "/kaggle/input/wav2lip-checkpoints/wav2lip_gan.pth"
S3FD_SRC = None  # e.g. "/kaggle/input/wav2lip-checkpoints/s3fd.pth"

CKPT_DST = "/kaggle/working/Wav2Lip/checkpoints/wav2lip_gan.pth"
S3FD_DST = "/kaggle/working/Wav2Lip/face_detection/detection/sfd/s3fd.pth"

CKPT_URL = "https://github.com/justinjohn0306/Wav2Lip/releases/download/models/wav2lip_gan.pth"
S3FD_URL = "https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth"

if CKPT_SRC:
    shutil.copy(CKPT_SRC, CKPT_DST)
else:
    urllib.request.urlretrieve(CKPT_URL, CKPT_DST)

if S3FD_SRC:
    shutil.copy(S3FD_SRC, S3FD_DST)
else:
    urllib.request.urlretrieve(S3FD_URL, S3FD_DST)

print("Checkpoints ready.")

## 3. Generate speech in your voice (XTTS-v2)

In [ ]:
from TTS.api import TTS

tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to("cuda")
tts.tts_to_file(
    text=SCRIPT_TEXT,
    speaker_wav=VOICE_REFERENCE,
    language="en",
    file_path=GENERATED_SPEECH,
)
print("Generated:", GENERATED_SPEECH)

## 4. Lip-sync the face clip to the new audio (Wav2Lip)

In [ ]:
%cd /kaggle/working/Wav2Lip
!python inference.py \
    --checkpoint_path checkpoints/wav2lip_gan.pth \
    --face "{FACE_CLIP}" \
    --audio "{GENERATED_SPEECH}" \
    --outfile "{OUTPUT_VIDEO}"

## 5. Preview

In [ ]:
from IPython.display import Video
Video(OUTPUT_VIDEO, embed=True, width=480)